# PS4 pipeline server (Colab A100) for the local frontend
Runs the hand-tracking pipeline + clip quality rating as an HTTP API on this Colab GPU and opens a
public HTTPS tunnel to it. Your frontend (on your computer) uploads 30 s–1 min clips to that URL and
reads the results.

**Before you start:** Runtime → Change runtime type → **A100**. Put `ps4v4.zip` (latest) in `MyDrive/mytronD/`.
Run cells 1–3, add `PS4_API_KEY` and `PS4_DISCOVERY_TOPIC` in Colab Secrets; use the same values in `dashboard/.env`. Keep this tab open while you use it.

*Data rule: uploaded clips and results stay on this Colab machine; cell 5 deletes them.*

In [ ]:
# CELL 1: install (~3 min)
!pip -q install ultralytics "smplx==0.1.28" timm einops roma scikit-image huggingface_hub pyyaml setuptools wheel
!pip -q install --no-deps git+https://github.com/warmshao/WiLoR-mini
!pip -q install --no-build-isolation --no-deps git+https://github.com/mattloper/chumpy
!pip -q install git+https://github.com/cvg/GeoCalib
!pip -q install fastapi "uvicorn[standard]" python-multipart
import importlib.util, pathlib
_p = pathlib.Path(importlib.util.find_spec("chumpy").submodule_search_locations[0])
for _f in _p.glob("*.py"):
    _s = _f.read_text(); _o = _s
    _s = _s.replace("from numpy import bool, int, float, complex, object, unicode, str, nan, inf", "from numpy import nan, inf")
    _s = _s.replace("inspect.getargspec", "inspect.getfullargspec")
    if _s != _o: _f.write_text(_s)
import chumpy, torch
print("install OK | GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU!")

In [ ]:
# CELL 2: code + model weights (~2 min)
from google.colab import drive
drive.mount("/content/drive")
import zipfile, shutil, os, glob
DRIVE_DIR = "/content/drive/MyDrive/mytronD"
def has(z, f):
    try: return any(n.endswith(f) for n in zipfile.ZipFile(z).namelist())
    except zipfile.BadZipFile: return False
zips = [z for z in sorted(glob.glob(f"{DRIVE_DIR}/*.zip"), key=os.path.getmtime, reverse=True) if has(z, "server/app.py")]
assert zips, "upload the latest ps4v4.zip (it must contain server/app.py) to " + DRIVE_DIR
shutil.rmtree("/content/_src", ignore_errors=True); shutil.rmtree("/content/ps4v4", ignore_errors=True)
zipfile.ZipFile(zips[0]).extractall("/content/_src")
root = os.path.dirname(sorted(glob.glob("/content/_src/**/run.py", recursive=True), key=len)[0])
shutil.move(root, "/content/ps4v4")
print("code from", zips[0])
!cd /content/ps4v4 && python tests/test_synthetic.py 2>&1 | tail -1
!cd /content/ps4v4 && python -c "from ps4v4.models.wilor_loader import load_wilor; load_wilor('cuda'); print('WiLoR weights OK')"
!python -c "from geocalib import GeoCalib; GeoCalib(); print('GeoCalib weights OK')"
!python -c "from transformers import CLIPModel, CLIPProcessor as P; CLIPModel.from_pretrained('openai/clip-vit-base-patch32'); P.from_pretrained('openai/clip-vit-base-patch32'); print('CLIP weights OK')"

In [ ]:
# CELL 3: start the API + a public HTTPS tunnel
import os, re, subprocess, time, json, urllib.request, threading
from pathlib import Path
from google.colab import userdata
try:
    API_KEY = userdata.get("PS4_API_KEY")
    DISCOVERY_TOPIC = userdata.get("PS4_DISCOVERY_TOPIC")
    assert API_KEY and DISCOVERY_TOPIC
except Exception as exc:
    raise RuntimeError("Add PS4_API_KEY and PS4_DISCOVERY_TOPIC in Colab Secrets and grant notebook access.") from exc
MAX_SECONDS = 90                                 # longer uploads are cut to this
PORT        = 8000
env = dict(os.environ, PS4_API_KEY=API_KEY, PS4_MAX_SECONDS=str(MAX_SECONDS), PS4_JOBS_DIR="/content/jobs",
           PS4_OVERRIDES=json.dumps({"camera": {"use_geocalib": True}}), PYTHONPATH="/content/ps4v4")
subprocess.run("pkill -f 'uvicorn server.app' ; pkill -f cloudflared", shell=True)
server = subprocess.Popen(["python", "-m", "uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", str(PORT)],
                          cwd="/content/ps4v4", env=env, stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("server:", urllib.request.urlopen(f"http://localhost:{PORT}/health").read().decode()); break
    except Exception:
        time.sleep(1)
if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m: URL = m.group(0); break
    time.sleep(1)
print("\n================ give these to the frontend ================")
print("API URL :", URL)
print("API KEY :", API_KEY[:4] + "… (from Secrets)")
print("test    :", f"{URL}/health")
print("=============================================================")
# Only the tunnel URL is published; the key never leaves Colab Secrets.
_publish_stop = threading.Event()
def publish_url():
    payload = json.dumps({"url": URL, "started": time.time(), "gpu": health.get("gpu")})
    try:
        Path("/content/drive/MyDrive/mytronD/api_url.json").write_text(payload)
    except OSError:
        print("Drive discovery write failed; using ntfy/manual URL")
    req = urllib.request.Request("https://ntfy.sh/" + DISCOVERY_TOPIC,
                                 data=payload.encode(), method="POST")
    try:
        urllib.request.urlopen(req, timeout=10).close()
    except Exception as exc:
        print("Discovery publish failed; Drive/manual URL still available:", type(exc).__name__)

def discovery_loop():
    global URL, tunnel, health
    last_publish = 0
    while not _publish_stop.is_set():
        try:
            restarted = False
            if tunnel.poll() is not None:
                restarted = True
                tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}",
                    "--no-autoupdate"], stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
                for _ in range(60):
                    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", Path("/content/tunnel.log").read_text())
                    if m:
                        URL = m.group(0)
                        break
                    time.sleep(1)
            if restarted or time.time() - last_publish >= 600:
                health = json.loads(urllib.request.urlopen(URL + "/health", timeout=5).read())
                publish_url()
                last_publish = time.time()
        except Exception as exc:
            print("Discovery waiting:", type(exc).__name__)
        _publish_stop.wait(5)

if not URL:
    raise RuntimeError("Tunnel did not start; inspect /content/tunnel.log")
health = json.loads(urllib.request.urlopen(URL + "/health", timeout=5).read())
threading.Thread(target=discovery_loop, daemon=True).start()


In [ ]:
# CELL 4 (optional): watch jobs and the server log
import json, urllib.request
req = urllib.request.Request(f"http://localhost:{PORT}/jobs", headers={"X-API-Key": API_KEY})
for j in json.loads(urllib.request.urlopen(req).read()):
    print(f"{j['id']}  {j['name'][:30]:30s} {j['status']:8s} {j.get('stage',''):26s} {100*j.get('progress',0):5.1f}%  "
          f"score={(j.get('quality') or {}).get('score')}")
!tail -n 15 /content/server.log

In [ ]:
_publish_stop.set() if "_publish_stop" in globals() else None
# CELL 5: stop and delete all uploaded clips and results (data rule: run this when you are done)
import shutil, subprocess
subprocess.run("pkill -f 'uvicorn server.app' ; pkill -f cloudflared", shell=True)
shutil.rmtree("/content/jobs", ignore_errors=True)
print("server stopped, all jobs deleted")

### Notes
- The URL changes every time cell 3 runs (free Cloudflare quick tunnel, no account). The local gateway discovers it automatically via ntfy/Drive; Settings supports a manual override.
- Upload limit through the tunnel is ~100 MB: fine for 30 s–1 min clips. A 1-min clip takes ~3–5 min on an A100.
- Jobs run one at a time; models stay loaded between jobs. If Colab disconnects, run cells 1–3 again.
- API reference for the frontend: `docs/API.md` in the code.